#### Check if vlr_address is populated in SS7 dataset

In [ ]:
import sys
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent if (Path.cwd() / "ingestion").exists() is False else Path.cwd()
sys.path.insert(0, str(project_root))

from ingestion.ss7 import clean

c:\Users\IshitaGodani\Documents\projects\spam-detection-prototype\notebooks


In [16]:
# pick one raw file (adjust path/glob as needed)
files = sorted((project_root / "data" / "raw" / "SS7").rglob("*.csv"))
print(f"{len(files)} files found")

raw = pd.read_csv(f, low_memory=False)

cleaned_parts = []
for i, f in enumerate(files, 1):
    raw = pd.read_csv(f, low_memory=False)
    c = clean(raw)
    c["file"] = f.name          # keep provenance - lets you break results down per file later
    cleaned_parts.append(c)
    print(f"[{i}/{len(files)}] {f.name}: {len(raw)} raw -> {len(c)} kept")

cleaned = pd.concat(cleaned_parts, ignore_index=True)
print(f"\nTotal kept rows across all files: {len(cleaned)}")

48 files found
  clean (SS7): 7 MO/MT_request row(s) have no decodable text - kept (not dropped), flagged via text_decode_failed
[1/48] stg_ss7_20260802_0000.csv: 221211 raw -> 49279 kept
  clean (SS7): 17 MO/MT_request row(s) have no decodable text - kept (not dropped), flagged via text_decode_failed
[2/48] stg_ss7_20260802_0100.csv: 183926 raw -> 38463 kept
  clean (SS7): 4 MO/MT_request row(s) have no decodable text - kept (not dropped), flagged via text_decode_failed
[3/48] stg_ss7_20260802_0200.csv: 172013 raw -> 32320 kept
  clean (SS7): 5 MO/MT_request row(s) have no decodable text - kept (not dropped), flagged via text_decode_failed
[4/48] stg_ss7_20260802_0300.csv: 168842 raw -> 31032 kept
  clean (SS7): 14 MO/MT_request row(s) have no decodable text - kept (not dropped), flagged via text_decode_failed
[5/48] stg_ss7_20260802_0400.csv: 152190 raw -> 28384 kept
  clean (SS7): 6 MO/MT_request row(s) have no decodable text - kept (not dropped), flagged via text_decode_failed
[6/4

In [18]:
populated = cleaned[cleaned["vlr_address"].notna()]

In [19]:
#cleaned how many
print(f"Total kept rows (MO+MT_request): {len(cleaned)}")
print(f"vlr_address populated: {len(populated)} ({100*len(populated)/len(cleaned):.2f}%)")
print()
print("By message_type:")
print(cleaned.groupby("message_type")["vlr_address"].apply(lambda s: s.notna().sum()))
print()

Total kept rows (MO+MT_request): 3417425
vlr_address populated: 896021 (26.22%)

By message_type:
message_type
2    896021
3         0
Name: vlr_address, dtype: int64



In [20]:
#records
cols = ["record_id", "message_type", "virtual_imsi", "vlr_address", "time_stamp"]
print(populated[cols].head(20).to_string(index=False))

                              record_id  message_type  virtual_imsi  vlr_address          time_stamp
0362199856756632_20260802000149_master1             2  5.021223e+14 6.012000e+10 2026-08-02 00:01:49
0221650994635317_20260802000020_master2             2  5.021228e+14 6.012000e+10 2026-08-02 00:00:20
0604062041980135_20260802000025_master1             2  5.021218e+14 6.012000e+10 2026-08-02 00:00:24
0489938582923687_20260802000136_master2             2  5.021216e+14 6.012000e+10 2026-08-02 00:01:35
0573643879623200_20260802000138_master2             2  5.021212e+14 6.012000e+10 2026-08-02 00:01:37
0034031759874310_20260802000136_master1             2  5.021237e+14 6.012000e+10 2026-08-02 00:01:35
0773706522845673_20260802000137_master1             2  5.021275e+14 6.012000e+10 2026-08-02 00:01:37
0945215739451624_20260802000139_master1             2  5.021224e+14 6.012000e+10 2026-08-02 00:01:38
0308428357366326_20260802000140_master1             2  5.021214e+14 6.012000e+10 2026-08-02

In [23]:
import pandas as pd
#why exactly is this failing? Let's try to diagnose the failure reason
def diagnose(row):
    content = row["content"]
    if not isinstance(content, str) or not content:
        return "empty/missing content"
    try:
        bytes.fromhex(content)
    except ValueError:
        return "invalid hex in content"
    return f"decode_by_dcs failed (dcs={row['dcs']})"

failed = cleaned[cleaned["text_decode_failed"]].copy()
print(len(failed))
failed["reason"] = failed.apply(diagnose, axis=1)
print(failed[["record_id", "message_type", "dcs", "content", "reason"]].to_string(index=False))


1620
                              record_id  message_type  dcs                                                                                                                                                                                                                                                                      content                         reason
0602523542945969_20260802000911_master1             3    8                                                                                                                                                                                                                                                                          NaN          empty/missing content
0240838042034775_20260802001337_master2             3    0                                                                                                                                                                                                                           